# TIN-17: Layer Sensitivity Profiling with Direction Injection
### Tiny Aya — Interpretability Analysis

**Goal:** Identify which transformer layers are most sensitive to language-specific
direction injection across all four Tiny Aya variants (Global, Water, Earth, Fire).

**Pipeline:**
1. Load FLORES+ parallel sentences via `src.data.flores_loader`
2. Compute a language direction per layer: `mean(lang) - mean(English)`
3. Inject the direction at each layer via a forward hook
4. Measure sensitivity: `1 - cosine_sim(injected, baseline)` at the final layer
5. Plot and compare sensitivity profiles across variants and languages

> **Before running:** Set runtime to GPU (T4). Runtime → Change runtime type → T4 GPU.

## 0. Login

In [ ]:
# Authenticate with HuggingFace.
# The src.data.flores_loader will load HF_TOKEN from the .env file automatically.
# If running in Colab, add HF_TOKEN via the Secrets sidebar instead.
import os, sys

# Add repo root to path so src.* imports work
sys.path.insert(0, '..')

try:
    # Colab: load from Secrets
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    # Local: flores_loader will pick up HF_TOKEN from .env automatically
    print('Not in Colab — flores_loader will use .env for HF_TOKEN.')

## 1. Imports & Device Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM
import gc

# Repo data loader and language definitions
from src.data.flores_loader import load_flores_parallel_corpus
from src.utils.languages import Language

# Analysis utilities for this analysis
from src.analysis.layer_sensitivity_profiling.sensitivity_utils import (
    get_mean_hidden_states,
    compute_language_directions,
    make_injection_hook,
    measure_sensitivity_batched,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# Model variants
MODEL_VARIANTS = {
    'Global': 'CohereLabs/tiny-aya-global',
    'Water':  'CohereLabs/tiny-aya-water',
    'Earth':  'CohereLabs/tiny-aya-earth',
    'Fire':   'CohereLabs/tiny-aya-fire',
}

# Languages to analyse — subset of Language enum matching project scope
# South Asia: Hindi, Bengali, Tamil
# Africa: Swahili, Amharic, Yoruba
# West Asia: Arabic, Turkish, Persian
# Europe: German, French, Spanish
TARGET_LANGUAGES = [
    Language.HINDI, Language.BENGALI, Language.TAMIL,
    Language.SWAHILI, Language.AMHARIC, Language.YORUBA,
    Language.ARABIC, Language.TURKISH, Language.PERSIAN,
    Language.GERMAN, Language.FRENCH, Language.SPANISH,
]
ENGLISH = Language.ENGLISH

# Region mapping for plot colouring
LANGUAGE_REGIONS = {
    'hindi': 'South Asia', 'bengali': 'South Asia', 'tamil': 'South Asia',
    'swahili': 'Africa', 'amharic': 'Africa', 'yoruba': 'Africa',
    'arabic': 'West Asia', 'turkish': 'West Asia', 'persian': 'West Asia',
    'german': 'Europe', 'french': 'Europe', 'spanish': 'Europe',
}
REGION_COLORS = {
    'South Asia': '#d62728',
    'Africa':     '#2ca02c',
    'West Asia':  '#ff7f0e',
    'Europe':     '#1f77b4',
}

# Analysis settings
N_INJECTION_SENTENCES = 20    # sentences for the injection sweep
INJECTION_SCALE       = 20.0  # direction injection strength (alpha)
MAX_LENGTH            = 64    # token limit per sentence

## 3. Load FLORES+ Dataset

In [ ]:
# Load parallel corpus using the shared repo data loader.
# Returns dict[lang_name, list[str]] with 1,012 aligned sentences per language.
print('Loading FLORES+ corpus...')
corpus = load_flores_parallel_corpus(
    languages=[ENGLISH] + TARGET_LANGUAGES
)

# Split into direction sentences (all) and injection sentences (small subset)
direction_corpus = corpus
injection_corpus = {lang: sents[:N_INJECTION_SENTENCES]
                    for lang, sents in corpus.items()}

print(f'Loaded {len(list(corpus.values())[0])} sentences per language '
      f'for {len(corpus)} languages.')

## 4. Run the Analysis

For each model variant: load, compute directions, sweep injection across all layers, unload VRAM.

**Estimated runtime on T4:** ~1.5 hours total.

In [ ]:
# results[variant_name][lang_name] = np.ndarray of shape (n_layers,)
results = defaultdict(dict)

for variant_name, model_id in MODEL_VARIANTS.items():
    print('\n' + '='*60)
    print(f'Model variant: {variant_name} ({model_id})')
    print('='*60)

    print('Loading model and tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map='auto'
    )
    model.eval()
    n_layers = len(model.model.layers)
    print(f'Model loaded. Transformer layers: {n_layers}')

    print('\nComputing language directions...')
    directions = compute_language_directions(
        direction_corpus, ENGLISH.lang_name, model, tokenizer, device
    )

    print('\nRunning injection sweep...')
    from tqdm.auto import tqdm
    for lang in tqdm(TARGET_LANGUAGES, desc='Languages'):
        lang_name = lang.lang_name
        sensitivity = measure_sensitivity_batched(
            injection_corpus[lang_name],
            directions[lang_name],
            model, tokenizer, device,
            max_length=MAX_LENGTH,
            scale=INJECTION_SCALE
        )
        results[variant_name][lang_name] = sensitivity
        peak  = np.argmax(sensitivity)
        score = np.max(sensitivity)
        print(f'  {lang_name}: peak at layer {peak} (score={score:.4f})')

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f'\nFinished {variant_name}. VRAM freed.')

print('\nAll variants complete!')

## 5. Save Raw Results

In [ ]:
import json

results_serialisable = {
    variant: {lang: scores.tolist() for lang, scores in langs.items()}
    for variant, langs in results.items()
}

with open('sensitivity_results.json', 'w') as f:
    json.dump(results_serialisable, f, indent=2)

print('Results saved to sensitivity_results.json')

## 6. Visualisation

### 6a. Per-variant: Sensitivity curves coloured by region

In [ ]:
def plot_variant_sensitivity(variant_name, lang_results, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    region_line_styles = ['-', '--', ':']
    region_lang_count  = defaultdict(int)

    for lang_name, scores in lang_results.items():
        region    = LANGUAGE_REGIONS.get(lang_name, 'Other')
        color     = REGION_COLORS.get(region, 'gray')
        linestyle = region_line_styles[region_lang_count[region] % len(region_line_styles)]
        region_lang_count[region] += 1
        n = len(scores)
        ax.plot(range(n), scores, label=lang_name, color=color,
                linestyle=linestyle, linewidth=1.8, alpha=0.9)

    ax.set_title(variant_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Injection Layer', fontsize=11)
    ax.set_ylabel('Sensitivity (1 - cosine sim)', fontsize=11)
    ax.legend(fontsize=8, loc='upper left', ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, n - 1)


fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for ax, (variant_name, lang_results) in zip(axes, results.items()):
    plot_variant_sensitivity(variant_name, lang_results, ax=ax)

legend_elements = [Line2D([0], [0], color=c, linewidth=2, label=r)
                   for r, c in REGION_COLORS.items()]
fig.legend(handles=legend_elements, title='Region', loc='lower center',
           ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.03))
fig.suptitle('Layer Sensitivity Profiling with Direction Injection\nTiny Aya - All Variants',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sensitivity_by_variant.png', dpi=150, bbox_inches='tight')
plt.show()

### 6b. Per-language: Compare sensitivity across all 4 variants

In [ ]:
variant_colors = {'Global': '#1f77b4', 'Water': '#17becf', 'Earth': '#8c564b', 'Fire': '#d62728'}

lang_names = [l.lang_name for l in TARGET_LANGUAGES]
ncols = 3
nrows = int(np.ceil(len(lang_names) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for idx, lang_name in enumerate(lang_names):
    ax     = axes[idx]
    region = LANGUAGE_REGIONS.get(lang_name, '')
    for variant_name, lang_results in results.items():
        if lang_name in lang_results:
            ax.plot(range(len(lang_results[lang_name])), lang_results[lang_name],
                    label=variant_name, color=variant_colors.get(variant_name, 'gray'),
                    linewidth=2)
    ax.set_title(f'{lang_name}  [{region}]', fontsize=11, fontweight='bold')
    ax.set_xlabel('Injection Layer', fontsize=9)
    ax.set_ylabel('Sensitivity', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(len(lang_names), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('Per-Language Sensitivity: Global vs Regional Variants',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('sensitivity_by_language.png', dpi=150, bbox_inches='tight')
plt.show()

### 6c. Heatmap: Peak sensitivity layer per language x variant

In [ ]:
variant_names = list(MODEL_VARIANTS.keys())
lang_names    = [l.lang_name for l in TARGET_LANGUAGES]

peak_matrix = np.zeros((len(lang_names), len(variant_names)))
for j, variant in enumerate(variant_names):
    for i, lang_name in enumerate(lang_names):
        if lang_name in results[variant]:
            scores = results[variant][lang_name]
            peak_matrix[i, j] = np.argmax(scores) / (len(scores) - 1)

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(peak_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Normalised peak sensitivity layer (0=early, 1=late)')
ax.set_xticks(range(len(variant_names)))
ax.set_xticklabels(variant_names, fontsize=11)
ax.set_yticks(range(len(lang_names)))
ylabels = [f'{n}  ({LANGUAGE_REGIONS.get(n, "")})' for n in lang_names]
ax.set_yticklabels(ylabels, fontsize=10)
ax.set_title('Peak Sensitivity Layer (Normalised by Model Depth)', fontsize=13, fontweight='bold')
for i in range(len(lang_names)):
    for j in range(len(variant_names)):
        ax.text(j, i, f'{peak_matrix[i, j]:.2f}', ha='center', va='center',
                color='black' if peak_matrix[i, j] < 0.6 else 'white', fontsize=9)
plt.tight_layout()
plt.savefig('peak_layer_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary Table

In [ ]:
lang_names    = [l.lang_name for l in TARGET_LANGUAGES]
variant_names = list(MODEL_VARIANTS.keys())

header = f'  Language         Region          ' + ''.join(f'{v:>16}' for v in variant_names)
print('=' * 95)
print(header)
print('  (peak layer / max score)')
print('-' * 95)

for lang_name in lang_names:
    region = LANGUAGE_REGIONS.get(lang_name, '')
    row = f'  {lang_name:<16}  {region:<14}  '
    for variant in variant_names:
        if lang_name in results[variant]:
            scores = results[variant][lang_name]
            row += f'   L{np.argmax(scores):02d}/{np.max(scores):.3f}   '
        else:
            row += f'      N/A        '
    print(row)

print('=' * 95)

## 8. Download Results

In [ ]:
try:
    from google.colab import files
    files.download('sensitivity_results.json')
    files.download('sensitivity_by_variant.png')
    files.download('sensitivity_by_language.png')
    files.download('peak_layer_heatmap.png')
except ImportError:
    print('Not in Colab — files saved to current directory.')